# Held-out counterfactual pairs — per-source breakdown

Recomputes the **pair-based counterfactual discrimination metrics** (Section 4.4 /
Table 18 of the paper) for the main fine-tuned BERT (Table 4), now stratified by
the dataset source. The current paper reports a single aggregate PairAcc=78.2%,
DirAcc=97.3%, MeanGap=0.705 on the 983 held-out pairs. This notebook splits that
into `biased-corpus` / `gemini` / `gus-dataset` pairs to check whether
discrimination depends on the source of the pair (a natural source-aware
robustness check that ties directly to the existing artifact analysis).

Metrics, identical to Section 4.4:
- **PairAcc**  $= \frac{1}{M}\sum \mathbf{1}\{\hat{y}(b_i){=}1 \wedge \hat{y}(n_i){=}0\}$
- **DirAcc**   $= \frac{1}{M}\sum \mathbf{1}\{p(b_i){>}p(n_i)\}$
- **MeanGap**  $= \frac{1}{M}\sum (p(b_i){-}p(n_i))$

Reported as mean $\pm$ std over the 5 seeds.

In [ ]:
!pip -q install transformers

In [ ]:
import gc, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT     = Path('/content/drive/MyDrive/attention-atlas-colab')
OUT_DIR  = ROOT / 'heldout_pairs_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Locate the main BERT checkpoints + v9 dataset + persisted split file.
CKPT_CANDIDATES = [
    ROOT / 'bert_baseline_outputs',
    ROOT / 'attention-atlas' / 'dataset' / 'v2' / 'bert_baseline_outputs',
    ROOT / 'dataset' / 'v2' / 'bert_baseline_outputs',
    Path('/content/attention-atlas/dataset/v2/bert_baseline_outputs'),
]
CKPT_DIR = next((c for c in CKPT_CANDIDATES if c.exists() and any(c.glob('bert_seed_*'))), None)
assert CKPT_DIR is not None, f'No bert_seed_* checkpoints found. Tried: {CKPT_CANDIDATES}'
print(f'CKPT_DIR : {CKPT_DIR}')

DATA_CANDIDATES = [
    ROOT / 'bias_sentences_v9.json',
    ROOT / 'attention-atlas' / 'dataset' / 'bias_sentences_v9.json',
    ROOT / 'dataset' / 'bias_sentences_v9.json',
    Path('/content/attention-atlas/dataset/bias_sentences_v9.json'),
]
DATA_JSON = next((p for p in DATA_CANDIDATES if p.exists()), None)
assert DATA_JSON is not None, f'bias_sentences_v9.json not found. Tried: {DATA_CANDIDATES}'
print(f'DATA_JSON: {DATA_JSON}')

SPLIT_CANDIDATES = [
    ROOT / 'bert_bias_classifier_v9_split.npz',
    ROOT / 'attention-atlas' / 'attention_app' / 'bias' / 'models' / 'bert_bias_classifier_v9_split.npz',
    Path('/content/attention-atlas/attention_app/bias/models/bert_bias_classifier_v9_split.npz'),
]
SPLIT_NPZ = next((p for p in SPLIT_CANDIDATES if p.exists()), None)
assert SPLIT_NPZ is not None, f'bert_bias_classifier_v9_split.npz not found. Tried: {SPLIT_CANDIDATES}'
print(f'SPLIT_NPZ: {SPLIT_NPZ}')

SEEDS = [1, 2, 3, 4, 5]
def ckpt_main(seed):
    return CKPT_DIR / f'bert_seed_{seed}'

## 1. Load v9 and identify the held-out counterfactual pairs

Held-out pairs = counterfactuals whose original member is in `test_idx`. The
main BERT was trained without these counterfactuals seeing training, so they
form the natural set for pair-based discrimination evaluation.

In [ ]:
with open(DATA_JSON, encoding='utf-8') as f:
    raw = json.load(f)
df = pd.DataFrame(raw['entries']).copy()
df['label'] = df['has_bias'].astype(int)

SOURCE_CANONICAL = {
    'biased_corpus_only': 'biased-corpus',
    'biased_corpus_v2':   'biased-corpus',
    'gemini_only':        'gemini',
    'gemini_only_v2':     'gemini',
    'gus_only':           'gus-dataset',
    'gus_only_v2':        'gus-dataset',
}
df['source_canonical'] = df['source'].map(SOURCE_CANONICAL).fillna(df['source'])

split = np.load(SPLIT_NPZ)
test_idx = split['test_idx']
test_pair_ids = set(df.iloc[test_idx]['pair_id'].dropna())
print(f'test_idx size: {len(test_idx)}  |  unique pair_ids in test: {len(test_pair_ids)}')

# Build paired rows: for each test original, find its counterfactual partner
test_originals = df.iloc[test_idx][['id', 'pair_id', 'source_canonical', 'text', 'has_bias']]
test_originals = test_originals[test_originals['pair_id'].notna()]

cf_pool = df[(df['role'] == 'counterfactual') & (df['pair_id'].isin(test_pair_ids))]
cf_pool = cf_pool.drop_duplicates(subset='pair_id', keep='first')

pairs = test_originals.merge(
    cf_pool[['pair_id', 'id', 'text', 'has_bias']],
    on='pair_id', suffixes=('_orig', '_cf')
)

# Tag biased and neutral members of each pair
def split_pair(row):
    if bool(row['has_bias_orig']):
        return pd.Series({'biased_text': row['text_orig'], 'biased_id': row['id_orig'],
                          'neutral_text': row['text_cf'],  'neutral_id': row['id_cf']})
    return pd.Series({'biased_text': row['text_cf'],  'biased_id': row['id_cf'],
                      'neutral_text': row['text_orig'], 'neutral_id': row['id_orig']})

pairs = pairs.join(pairs.apply(split_pair, axis=1))
pairs = pairs.rename(columns={'source_canonical': 'source'})
pairs = pairs[['pair_id', 'source', 'biased_id', 'biased_text', 'neutral_id', 'neutral_text']]
pairs = pairs.reset_index(drop=True)

print(f'\nHeld-out counterfactual pairs: {len(pairs)}')
print('By source:')
print(pairs['source'].value_counts().to_string())

## 2. Inference helpers and encoding

In [ ]:
MODEL_ID = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 32
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def encode(texts):
    enc = tokenizer(
        list(texts), padding='max_length', truncation=True,
        max_length=MAX_LEN, return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']

def predict_probs(model, ids, mask):
    model.eval()
    dset = TensorDataset(ids.to(device), mask.to(device))
    loader = DataLoader(dset, batch_size=BATCH_SIZE)
    out = []
    with torch.no_grad():
        for b_ids, b_mask in loader:
            logits = model(input_ids=b_ids, attention_mask=b_mask).logits
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

ids_b, mask_b = encode(pairs['biased_text'].values)
ids_n, mask_n = encode(pairs['neutral_text'].values)
print(f'biased  : {tuple(ids_b.shape)}')
print(f'neutral : {tuple(ids_n.shape)}')

## 3. Inference across the 5 seeds

In [ ]:
rows = []
for seed in SEEDS:
    ckpt = ckpt_main(seed)
    if not ckpt.exists():
        print(f'[SKIP] seed {seed}: missing checkpoint at {ckpt}')
        continue
    print(f'seed {seed}: loading {ckpt.name} ...')
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device)
    p_b = predict_probs(model, ids_b, mask_b)
    p_n = predict_probs(model, ids_n, mask_n)

    for i in range(len(pairs)):
        rows.append({
            'pair_id':     int(pairs['pair_id'].iloc[i]),
            'seed':        seed,
            'source':      pairs['source'].iloc[i],
            'biased_id':   int(pairs['biased_id'].iloc[i]),
            'neutral_id':  int(pairs['neutral_id'].iloc[i]),
            'p_biased':    float(p_b[i]),
            'p_neutral':   float(p_n[i]),
        })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print(f'  done.  mean p_biased={p_b.mean():.4f}  mean p_neutral={p_n.mean():.4f}')

df_pred = pd.DataFrame(rows)
df_pred.to_csv(OUT_DIR / 'heldout_pairs_predictions.csv', index=False)
print(f'\nSaved {len(df_pred)} rows  ({len(pairs)} pairs × {len(SEEDS)} seeds)')

## 4. Aggregate metrics (sanity check against Table 18 = 78.2% / 97.3% / 0.705)

In [ ]:
def compute_metrics(group, threshold=0.5):
    pb = group['p_biased'].values
    pn = group['p_neutral'].values
    pair_acc = float((((pb >= threshold).astype(int) == 1) & ((pn >= threshold).astype(int) == 0)).mean())
    dir_acc  = float((pb > pn).mean())
    mean_gap = float((pb - pn).mean())
    return pd.Series({'PairAcc': pair_acc, 'DirAcc': dir_acc, 'MeanGap': mean_gap, 'n': int(len(group))})

per_seed = df_pred.groupby('seed').apply(compute_metrics).round(4)
print('Per-seed (aggregate, 983 pairs):')
print(per_seed.to_string())
per_seed.to_csv(OUT_DIR / 'heldout_pairs_per_seed.csv')

print('\nAggregate mean ± std over seeds (sanity check vs Table 18):')
for m in ['PairAcc', 'DirAcc', 'MeanGap']:
    mu, sd = per_seed[m].mean(), per_seed[m].std()
    print(f'  {m:<8}: {mu:.4f} ± {sd:.4f}')

## 5. Per-source breakdown (the new analysis)

In [ ]:
per_seed_src = (
    df_pred.groupby(['seed', 'source'])
           .apply(compute_metrics)
           .reset_index()
)
summary_src = (
    per_seed_src.groupby('source')[['PairAcc', 'DirAcc', 'MeanGap']]
                .agg(['mean', 'std'])
                .round(4)
)
summary_src['n_pairs'] = df_pred.groupby('source')['pair_id'].nunique()

print('Per-source PairAcc / DirAcc / MeanGap (mean ± std across 5 seeds):')
print(summary_src.to_string())
summary_src.to_csv(OUT_DIR / 'heldout_pairs_per_source.csv')

## 6. Files saved

In [ ]:
for f in sorted(OUT_DIR.glob('*.csv')):
    print(' -', f.name, f'({f.stat().st_size//1024} KB)')